## Gene analysis for fig 1

In [1]:
import pandas as pd
import glob
import re

data_dir = '/private/groups/migalab/juklucas/censat_regions/censat_arrays/tables/censat_regions_pass_qc_by_chrom'

dfs = []
for f in glob.glob(f'{data_dir}/censat_regions_pass_qc_chr*.tsv'):
    chrom = re.search(r'chr[\dXYM]+', f).group(0)
    df = pd.read_csv(f, sep='\t')
    df['chrom'] = chrom
    dfs.append(df)

censat_df = pd.concat(dfs, ignore_index=True)
print(censat_df.shape)
censat_df.head()


(4147, 26)


,sample_id,haplotype,assembly_id,sequence_id,region_start,region_end,region_size_bp,feature_count,region_label_bp_json,simplified_label_bp_json,...,contig_size,dist_to_contig_start,dist_to_contig_end,region_not_near_contig_edge,reference_large_array_labels_json,all_large_arrays_represented,sequences_per_assembly_chrom,single_sequence_for_chrom,pass_qc,chrom
0,CHM13,0,chm13v2.0_maskedY_rCRS,CHM13#0#chr18,14298107,21125235,6827128,33,"{""active_hor(S2C18H1L)"": 4967510, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6819, ""active"": 496751...",...,80542538.0,14298107,59417303.0,True,"[""active""]",True,1.0,True,True,chr18
1,HG00097,1,HG00097_hap1_hprc_r2_v1.0.1,HG00097#1#JBIRDD010000010.1,14135694,19270914,5135220,40,"{""active_hor(S2C18H1L)"": 3113600, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 311360...",...,78710895.0,14135694,59439981.0,True,"[""active""]",True,1.0,True,True,chr18
2,HG00097,2,HG00097_hap2_hprc_r2_v1.0.1,HG00097#2#CM094084.1,14138081,20853464,6715383,33,"{""active_hor(S2C18H1L)"": 4697016, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 469701...",...,80317024.0,14138081,59463560.0,True,"[""active""]",True,1.0,True,True,chr18
3,HG00099,1,HG00099_hap1_hprc_r2_v1.0.1,HG00099#1#CM087325.1,14150536,20957234,6806698,37,"{""active_hor(S2C18H1L)"": 4765242, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 476524...",...,80476601.0,14150536,59519367.0,True,"[""active""]",True,1.0,True,True,chr18
4,HG00099,2,HG00099_hap2_hprc_r2_v1.0.1,HG00099#2#CM087369.1,14095274,20715119,6619845,37,"{""active_hor(S2C18H1L)"": 4536062, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 453606...",...,80141396.0,14095274,59426277.0,True,"[""active""]",True,1.0,True,True,chr18


In [2]:
print(censat_df.columns.tolist())


['sample_id', 'haplotype', 'assembly_id', 'sequence_id', 'region_start', 'region_end', 'region_size_bp', 'feature_count', 'region_label_bp_json', 'simplified_label_bp_json', 'contains_active_array', 'observed_large_array_count', 'observed_large_array_labels_json', 'has_multi_source_qc_flag', 'chrom_assignment', 'level', 'contig_size', 'dist_to_contig_start', 'dist_to_contig_end', 'region_not_near_contig_edge', 'reference_large_array_labels_json', 'all_large_arrays_represented', 'sequences_per_assembly_chrom', 'single_sequence_for_chrom', 'pass_qc', 'chrom']


In [ ]:
cat_index=/private/home/juklucas/github/censat_paper/data_tables/annotation/genes/cat_genes_hprc_r2_v1.3.index.csv

### Pull all genes inside censat QC regions 

In [6]:
from pathlib import Path
import gzip


CAT_GENE_INDEX = Path("/private/home/juklucas/github/censat_paper/data_tables/annotation/genes/cat_genes_hprc_r2_v1.3.index.csv")

cat_gene_idx = pd.read_csv(CAT_GENE_INDEX)
cat_gene_idx["haplotype"] = cat_gene_idx["haplotype"].astype(str)
cat_gene_idx["sample_hap"] = cat_gene_idx["sample_id"].astype(str) + "." + cat_gene_idx["haplotype"]

## Helper functions 
def parse_gff3_attributes(attr_text):
    attrs = {}
    for field in str(attr_text).strip().split(";"):
        if not field or "=" not in field:
            continue
        key, value = field.split("=", 1)
        attrs[key] = value
    return attrs


def _clean_attr_value(value):
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    if not value or value in {".", "NA", "nan", "None"}:
        return None
    return value


def normalize_gene_token(value):
    value = _clean_attr_value(value)
    if value is None:
        return None

    value = value.split(",")[0].strip()
    value = re.sub(r"^(gene:|transcript:)", "", value)
    value = re.sub(r"^(ENS[A-Z]*\\d+)(?:\\.\\d+)?(?:_\\d+)?$", r"\\1", value)
    return value


def choose_collapsed_gene_name(attrs):
    candidates = [
        attrs.get("source_gene_common_name"),
        attrs.get("gene_name"),
        attrs.get("Name"),
        attrs.get("source_gene"),
        attrs.get("original_gene_id"),
        attrs.get("gene_id"),
        attrs.get("ID"),
    ]
    normalized = [normalize_gene_token(value) for value in candidates]
    normalized = [value for value in normalized if value]
    return normalized[0] if normalized else None


def get_cat_gff3_path(region_row, gene_index=cat_gene_idx):
    match = gene_index.loc[gene_index["assembly_name"] == region_row["assembly_name_combined"]]
    if match.empty:
        match = gene_index.loc[gene_index["sample_hap"] == region_row["sample_hap"]]
    if match.empty:
        raise ValueError(
            f"No CAT gene annotation found for {region_row['sample_hap']} / {region_row['assembly_name_combined']}"
        )
    return Path(match.iloc[0]["location"])


def normalize_cat_seqid(seqid):
    seqid = _clean_attr_value(seqid)
    if seqid is None:
        return None
    return str(seqid).split("#")[-1]

def iter_cat_gff3_features(gff3_path, sequence_id, interval_start, interval_end, feature_types=("gene", "transcript")):
    opener = gzip.open if str(gff3_path).endswith(".gz") else open
    normalized_sequence_id = normalize_cat_seqid(sequence_id)

    with opener(gff3_path, "rt") as fh:
        for line in fh:
            if not line or line.startswith("#"):
                continue

            parts = line.rstrip("\n").split("\t")
            if len(parts) != 9:
                continue

            seqid, source, feature_type, start, end, score, strand, phase, attrs = parts
            normalized_seqid = normalize_cat_seqid(seqid)
            if normalized_seqid != normalized_sequence_id:
                continue

            if feature_type not in feature_types:
                continue

            start0 = int(start) - 1
            end0 = int(end)
            if start0 >= interval_end or end0 <= interval_start:
                continue

            yield {
                "seqid": seqid,
                "source": source,
                "feature_type": feature_type,
                "start": start0,
                "end": end0,
                "strand": strand,
                "attrs": parse_gff3_attributes(attrs),
            }


In [7]:
from pathlib import Path

results = []

for _, region_row in censat_df.iterrows():
    sample_id = str(region_row['sample_id'])
    haplotype = str(region_row['haplotype'])

    match = cat_gene_idx[
        (cat_gene_idx['sample_id'] == sample_id) &
        (cat_gene_idx['haplotype'].astype(str) == haplotype)
    ]
    if match.empty:
        continue
    gff3_path = Path(match.iloc[0]['location'])

    for record in iter_cat_gff3_features(
        gff3_path=gff3_path,
        sequence_id=region_row['sequence_id'],
        interval_start=int(region_row['region_start']),
        interval_end=int(region_row['region_end']),
        feature_types=('gene',),
    ):
        attrs = record['attrs']
        gene_name = choose_collapsed_gene_name(attrs)

        # Skip bare Ensembl IDs
        if not gene_name or gene_name.upper().startswith('ENS'):
            continue

        results.append({
            'sample_id':    sample_id,
            'haplotype':    haplotype,
            'chrom':        region_row['chrom'],
            'sequence_id':  region_row['sequence_id'],
            'region_start': region_row['region_start'],
            'region_end':   region_row['region_end'],
            'gene_name':    gene_name,
            'gene_biotype': _clean_attr_value(attrs.get('gene_biotype')),
            'feature_start': record['start'],
            'feature_end':   record['end'],
            'strand':        record['strand'],
        })

censat_genes_df = pd.DataFrame(results)
print(censat_genes_df.shape)
censat_genes_df.head()


KeyboardInterrupt: 